# 02 — Dataset Assembly

## Section 0 — Setup

Imports, `ROOT_FOLDER` assert, `sys.path` bootstrap, `RANDOM_SEED` = 42, `CALIBRATION_MODE`, `REBUILD_FEATURES`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import subprocess
import json
import sys

CURRENT_FOLDER = Path.cwd()
ROOT_FOLDER = CURRENT_FOLDER.parent     # both stored as a Path not str
assert (ROOT_FOLDER / 'main.py').exists(), f"repo root is not {ROOT_FOLDER}"

root_str = str(ROOT_FOLDER)
if root_str not in sys.path:
    sys.path.insert(0, root_str)

from core.features import to_grid, max_disp, strain, velocity

RANDOM_SEED = 42
REBUILD_FEATURES = True
CALIBRATION_MODE = "per_wear"

## Section 1 — Session Discovery

Walk `recordings/`, return every participant and their usable sessions. The exclusion rule lives in one predicate.

In [ ]:
""" Participant discovery then Session discovery

participants is a dict: participant id -> sorted list of usable session numbers.

participant_num     -> pulls integer out of folder name like
                       'particpant001_...'. Will be returned
                       as participant and their useable sessions
                       as a dictionary

session_num         -> pulls integer out of every session folder
                       like 'session10_...'.

useable_sessions    -> session folders that actually carry a training_table.csv
                       glob('session*') is what keeps aborted/ folder out
                       no switching to iterdir() or rglob()

Task: Feed 
"""
recordings_dir = ROOT_FOLDER / 'recordings'

def participant_num(participant_dir):       # loads participant number, int strips leading 00
    removed_participant = participant_dir.name.removeprefix('participant')
    return int(removed_participant.split('_')[0])

def session_num(session_dir):
    removed_session = session_dir.name.removeprefix('session')
    return int(removed_session.split('_')[0])

def usable_sessions(participant_dir):
    return sorted(
        session_num(folder)
        for folder in participant_dir.glob('session*')
        if (folder / 'training_table.csv').exists()
    )

In [ ]:
"""Loop through recordings/ folder
Gets participant_num and usable sessions, store in dict
Should produce a dictionary 'participants' with shape: int -> sorted list of int
OUTPUT:     participants = {
                1: [1, 2, 3, 5, 6, 7, 8, 9, 10],   # session 4 excluded
                2: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                3: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                4: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                5: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                6: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                7: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            }
"""

participants = {}

for participant_dir in recordings_dir.glob('participant*_*'):
    if participant_dir.is_dir(): 
        p_num = participant_num(participant_dir)                # Gets participant number -> p_num

        participants[p_num] = usable_sessions(participant_dir) 

print(participants)
    

In [ ]:
# --- §1 acceptance ---
# FROZEN_PARTICIPANT_COUNT names the original 7-participant cohort this notebook was
# first built and frozen against -- EXPECTED_ROWS, TOTAL_ROWS_BEFORE/DROPPED, and the
# manifest totals downstream all guard THAT cohort against silently changing. A
# participant discovered beyond it gets a live self-consistency check instead of a
# frozen literal -- there is nothing yet to regress-test them against.
FROZEN_PARTICIPANT_COUNT = 7

# Contiguous numbering from 1, whatever N turns out to be -- generalizes for free.
assert sorted(participants) == list(range(1, len(participants) + 1)), sorted(participants)

if len(participants) >= FROZEN_PARTICIPANT_COUNT:
    assert sorted(participants)[:FROZEN_PARTICIPANT_COUNT] == list(range(1, FROZEN_PARTICIPANT_COUNT + 1))

assert participants[1] == [1, 2, 3, 5, 6, 7, 8, 9, 10]   # session 4 excluded by name --
                                                          # a fact about participant 1, not a
                                                          # count, and true at any N

for pid, sess in participants.items():
    assert isinstance(pid, int), pid
    assert sess == sorted(set(sess)), pid
    assert all(isinstance(s, int) and s > 0 for s in sess), pid

print(f"{len(participants)} participants, {sum(len(v) for v in participants.values())} sessions")

## Section 2 — Session Loading

**Load one session** plus its grid layout. Assert 405 columns, layout `source == "geometry"`, 99 markers.

Section 1 only kept **ints**, so section 2 has to get back to a path

In [ ]:
""" Create Functions: 

session_dir(p, s)   ->  takes two ints from session 1, returns the folder.
                        participant is zero-padded to 3 (participant003_*), 
                        session is not padded (session3_*). 

load_session(p, s)  ->  reads participant00x/sessionX's training_table.csv
                        returns a DataFrame

load_layout(p, s)   ->  reads grid_layout.json, returns parsed dict

"""

def session_dir(p, s):
    p_folder = f"participant{p:03d}_*"
    s_folder = f"session{s}_*"

    p_hits = list(recordings_dir.glob(p_folder))       # list, not next() -> a 2nd hit is visible
    assert len(p_hits) == 1, (p, p_hits)
    p_dir = p_hits[0]

    s_hits = list(p_dir.glob(s_folder))
    assert len(s_hits) == 1, (p, s, s_hits)

    return s_hits[0]                                   # already absolute, do not join to p_dir

def load_session(p, s):
    sdir = session_dir(p, s)                           # sdir, not session_dir -> no shadowing
    df = pd.read_csv(sdir / 'training_table.csv')
    return df

def load_layout(p, s):
    sdir = session_dir(p, s)
    with (sdir / 'grid_layout.json').open() as f:
        meta = json.load(f)

    return meta

In [ ]:
df = load_session(3, 3)         # Check for p003, s3

print(f"Shape: {df.shape}\nFirst 9 cols: {df.columns[0:9].tolist()}\n")

layout = load_layout(3,3)

print(f"Source: {layout['source']}\nGrid Shape: {layout['grid_shape']}")
print(f"Locked Count: {layout['locked_count']}")
print(f"Length of layout: {len(layout['layout'])}")

In [ ]:
""" Sweep for all sessions, headers only

Why: avoiding very large data stored in RAM

Output is a dict:
    - keyed by (participant, session) holding n_rows

Assert the following:
    - every session has 405 cols and source == 'geometry'
"""

all_sessions = {}

for p, s in participants.items():
    for item in s:
        sdir = session_dir(p, item)
        df_cols = pd.read_csv(sdir / 'training_table.csv', nrows=0)
        df_rows = pd.read_csv(sdir / 'training_table.csv', usecols=[0])
        session_layout = load_layout(p, item)
        all_sessions[(p, item)] = {
            'n_cols': df_cols.shape[1],
            'n_rows': df_rows.shape[0],
            'source': session_layout['source']
        }

pd.DataFrame(all_sessions).T
    

In [ ]:
# layout['layout'] -> comparable, hashable object
# marker ids walked 0..98 in order: JSON key order is scrambled and would
# make two identical grids compare unequal.

def layout_key(layout):
    grid = layout['layout']
    return tuple((grid[str(i)]['row'], grid[str(i)]['col']) for i in range(99))


layouts_by_participant = {}     # p -> list of layout_key, one per session
reference_layouts = {}          # p -> that participant's first layout_key

for p, sessions in participants.items():
    layouts_by_participant[p] = [layout_key(load_layout(p, s)) for s in sessions]
    reference_layouts[p] = layouts_by_participant[p][0]


# --- cross-participant report: print only, never assert ---
# different people can wear the sensor at different rotations, so a
# difference here is a finding for the summary, not a failure.

groups = {}
for p, key in reference_layouts.items():
    groups.setdefault(key, []).append(p)

print(f"{len(groups)} distinct layout(s) across {len(reference_layouts)} participants")
for members in groups.values():
    print(f"  shared by participants {members}")


In [ ]:
# --- §2 acceptance ---
EXPECTED_ROWS = {1: 28261, 2: 43941, 3: 38233, 4: 38669,
                 5: 40622, 6: 37874, 7: 37462}   # frozen regression guard -- the original
                                                  # FROZEN_PARTICIPANT_COUNT cohort only

# the sweep: every discovered session, 405 columns each, every layout from geometry
expected_total_sessions = sum(len(v) for v in participants.values())
assert len(all_sessions) == expected_total_sessions, len(all_sessions)
for key, rec in all_sessions.items():
    assert rec['n_cols'] == 405, (key, rec['n_cols'])
    assert rec['source'] == 'geometry', (key, rec['source'])

# row counts, per participant -- exact match for the frozen cohort, a live sanity
# check (positive, non-empty) for anyone discovered after it
for p in participants:
    rows = sum(all_sessions[(p, s)]['n_rows'] for s in participants[p])
    if p in EXPECTED_ROWS:
        assert rows == EXPECTED_ROWS[p], (p, rows, EXPECTED_ROWS[p])
    assert rows > 0, p

if set(EXPECTED_ROWS) <= set(participants):
    assert sum(EXPECTED_ROWS.values()) == 265062   # the frozen cohort's own total, unaffected by new arrivals

# the probe session: 99 markers counted two ways, and 4 x 99 marker columns
probe = load_layout(3, 3)
assert len(probe['layout']) == 99, len(probe['layout'])
assert probe['grid_shape']['n_rows'] * probe['grid_shape']['n_cols'] == 99
for prefix in ('dx_', 'dy_', 'dA_', 'mag_'):
    n = sum(c.startswith(prefix) for c in df.columns)
    assert n == 99, (prefix, n)

# every session of a participant shares that participant's layout
for p, keys in layouts_by_participant.items():
    assert all(k == reference_layouts[p] for k in keys), p

print(f"§2 OK: {len(all_sessions)} sessions, "
      f"{sum(r['n_rows'] for r in all_sessions.values())} rows, "
      f"{len(groups)} distinct layout(s)")


## Section 03 — Cue Offset



In [ ]:
def cue_offset(session_dir, df):
    with (session_dir / 'events.json').open() as f:
        events = json.load(f)['events']             # loads events containing grip starts

    # cue dict - {trial_id: timestamp}
    cue = {e['trial_id']: e['ts'] for e in events if e['type'] == 'grip_start'}

    # NaN for any trial with no cue
    cue_ts = np.array([cue.get(t, np.nan) for t in df['trial_id'].to_numpy()])

    # Returns seconds until the "go" cue
    return (df['timestamp_ms'].to_numpy() - cue_ts) / 1000.0

In [ ]:
""" Attach to_cue, then drop frames with no usable force label.

to_cue is still attached -- Section 4's phase/braced flags need it. An
uncued trial is NOT dropped for that alone: its force and marker data are
still real, continuous recordings (verified on disk against several such
trials). Only pct_mvc_corr being NaN means there is no label to train
against -- that is the actual "outside sync coverage" case, and it is the
same exclusion rule already used at the session level, applied per frame.
"""

def drop_unlabeled(session_dir, df):
    df = df.assign(to_cue=cue_offset(session_dir, df))
    return df[df['pct_mvc_corr'].notna()].reset_index(drop=True)

In [ ]:
# Run drop_unlabeled across all sessions of all participants

rows_before = {}       # (p, s) -> row count before the filter
rows_after = {}        # (p, s) -> row count after
uncued_trials = []      # (p, s, trial_id) -- KEPT, but to_cue is NaN for every frame of it

for p, s in participants.items():
    for item in s:
        sdir = session_dir(p, item)
        df = load_session(p, item)

        kept = drop_unlabeled(sdir, df)

        rows_before[(p, item)] = len(df)
        rows_after[(p, item)] = len(kept)

        no_cue = kept.loc[kept['to_cue'].isna(), 'trial_id'].unique()
        uncued_trials += [(p, item, int(t)) for t in sorted(no_cue)]

print(f"rows: {sum(rows_before.values())} -> {sum(rows_after.values())}")
print(f"dropped (no force label): {sum(rows_before.values()) - sum(rows_after.values())}")
print(f"uncued trials kept, {len(uncued_trials)}: {uncued_trials}")

In [ ]:
# --- §3 acceptance ---
# Frozen snapshot facts for the original FROZEN_PARTICIPANT_COUNT-participant cohort.
# Checked only while that exact cohort is still the whole of `participants` -- once
# participant 8+ joins, these totals legitimately move and stop being a regression
# guard for anything.
TOTAL_ROWS_BEFORE = 265062
TOTAL_ROWS_DROPPED = 455        # frames with no usable force label
TOTAL_UNCUED_TRIALS = 14        # kept, but to_cue is NaN for every frame

if len(participants) == FROZEN_PARTICIPANT_COUNT:
    assert sum(rows_before.values()) == TOTAL_ROWS_BEFORE
    assert sum(rows_before.values()) - sum(rows_after.values()) == TOTAL_ROWS_DROPPED
    assert sum(rows_after.values()) == TOTAL_ROWS_BEFORE - TOTAL_ROWS_DROPPED
    assert len(uncued_trials) == TOTAL_UNCUED_TRIALS

print(f"§3 OK: {sum(rows_after.values())} rows kept, "
      f"{sum(rows_before.values()) - sum(rows_after.values())} dropped (no force label), "
      f"{len(uncued_trials)} uncued trials kept and flagged")

## Section 04 — Flags

phase, braced, carryover. No rows removed -- every judgement lives in a column.

In [ ]:
BRACED_SEC = 1.0          # window before the cue that counts as anticipatory bracing
CARRYOVER_TAIL_MS = 500   # tail of a grip trial used to test for an incomplete release
CARRYOVER_PCT = 10.0      # mean %MVC over that tail that marks the trial


def add_flags(df):
    """ Add phase, braced, carryover -- row count never changes.

    phase / braced -> per-frame, straight off to_cue.
    carryover      -> a per-trial verdict, broadcast back to every frame of
                      that trial (transform, not agg -- Section 3's figure).

    to_cue is NaN for the 14 uncued trials Section 3 kept. NaN >= 0 is
    False, so a plain np.where would silently call all of them 'pre_cue' --
    wrong, since 7 of those 14 show real grip force, up to 74% MVC. phase
    must come out NaN too: we do not know when the cue was, so we do not
    get to guess the phase.

    braced does NOT need the same fix. '(to_cue >= -1) & (to_cue < 0)' is
    already False on NaN by how comparisons work, and False here just means
    "no evidence of bracing" -- true, not a guess.
    """
    # astype(object) first -- np.where on strings alone gives a fixed-width
    # '<U8' array, and numpy refuses to mix that with a float NaN below.
    phase = np.where(df['to_cue'] >= 0, 'post_cue', 'pre_cue').astype(object)
    phase = np.where(df['to_cue'].isna(), np.nan, phase)      # overwrite the NaN rows

    braced = (df['to_cue'] >= -BRACED_SEC) & (df['to_cue'] < 0)

    # Tail of every trial, by its own last timestamp -- only reads
    # timestamp_ms, never to_cue, so this works for the uncued trials too.
    trial_end = df.groupby('trial_id')['timestamp_ms'].transform('max')
    in_tail = df['timestamp_ms'] >= trial_end - CARRYOVER_TAIL_MS

    # Rest trials (target 0) and the one unknown-target trial (NaN --
    # participant 006 session 4 trial 13, an orphan trial_id with no
    # assigned level) are both excluded on purpose. carryover only means
    # something for a trial that was actually instructed to grip.
    grip_trial = df['target_pct'].notna() & (df['target_pct'] != 0)

    tail_mean = df[in_tail & grip_trial].groupby('trial_id')['pct_mvc_corr'].mean()
    flagged = tail_mean.index[tail_mean > CARRYOVER_PCT]

    return df.assign(
        phase=phase,
        braced=braced,
        carryover=df['trial_id'].isin(flagged),
    )

In [ ]:
# Run add_flags across all sessions of all participants

phase_counts = {'pre_cue': 0, 'post_cue': 0}
phase_unknown = 0        # rows where phase is NaN -- the 14 uncued trials
braced_frames = 0
carryover_trials = []    # (p, s, trial_id)

for p, s in participants.items():
    for item in s:
        sdir = session_dir(p, item)
        df = load_session(p, item)

        kept = drop_unlabeled(sdir, df)
        flagged_df = add_flags(kept)

        assert len(flagged_df) == len(kept), (p, item)     # add_flags changes no rows
        # braced must never point at a row that isn't pre_cue -- including
        # NaN phase, since NaN == 'pre_cue' is False, same guard either way.
        assert (flagged_df.loc[flagged_df['braced'], 'phase'] == 'pre_cue').all(), (p, item)

        for ph, c in flagged_df['phase'].value_counts(dropna=False).items():
            if pd.isna(ph):
                phase_unknown += int(c)
            else:
                phase_counts[ph] += int(c)

        braced_frames += int(flagged_df['braced'].sum())

        trials = flagged_df.loc[flagged_df['carryover'], 'trial_id'].unique()
        carryover_trials += [(p, item, int(t)) for t in sorted(trials)]

print(f"phase: {phase_counts}, unknown (uncued): {phase_unknown}")
print(f"braced: {braced_frames} frames")
print(f"carryover: {len(carryover_trials)} trials")

In [ ]:
# --- §4 acceptance ---

# docs/logs/20260808_151738_trial_labelling_and_grip_carryover.md §2 --
# documented for participant 001 only, dated 2026-08-08. STALE: session 6's
# training_table.csv was regenerated 2026-08-18, and now trial 1 (target
# 50%) shows a tail mean of 10.12% MVC -- just over threshold, verified by
# hand against the raw CSV, zero null rows, not a code bug. 8 -> 9.
CARRYOVER_DOCUMENTED_P1 = {
    (1, 1, 0), (1, 1, 2), (1, 1, 5), (1, 1, 13),
    (1, 2, 10), (1, 5, 12), (1, 5, 13), (1, 8, 5),
    (1, 6, 1),
}

p1_carryover = {t for t in carryover_trials if t[0] == 1}
assert p1_carryover == CARRYOVER_DOCUMENTED_P1, p1_carryover

# every row is either a known phase or honestly unknown -- nothing missing, nothing double-counted
assert sum(phase_counts.values()) + phase_unknown == sum(rows_after.values())

print(f"§4 OK: phase {phase_counts}, {phase_unknown} unknown (14 uncued trials)")
print(f"braced: {braced_frames} frames")
print(f"carryover: {len(carryover_trials)} trials total, "
      f"participant 001's 9 (was 8, doc is stale) reproduced exactly")

## Section 05 — Calibration

Three modes: `none`, `per_wear`, `stale`. Runs before Section 06 so every derived channel inherits the correction.

In [ ]:
CALIBRATED = ('dx', 'dy', 'dA')          # mag is rebuilt from dx/dy, never offset directly
CALIBRATION_MODES = ('per_wear', 'none', 'stale')


def rest_offset(df):
    """rest_offset(df) -> dict[str, np.ndarray] -- per-marker resting level, one (99,) array per channel.

    Window is trial 0's pre-cue interval minus the braced second -- requires add_flags
    to have already run.
    """
    window = (df['trial_id'] == 0) & (df['phase'] == 'pre_cue') & (~df['braced'])

    return {
        c: df.loc[window, [f'{c}_{i}' for i in range(99)]].median().to_numpy()
        for c in CALIBRATED
    }


def apply_calibration(df, offset):
    """apply_calibration(df, offset) -> DataFrame -- subtract offset, rebuild mag.

    offset=None (mode 'none') returns df untouched, so the three modes differ only
    in the numbers subtracted, never in which rows or trials survive.
    """
    if offset is None:
        return df

    vals = {
        c: df[[f'{c}_{i}' for i in range(99)]].to_numpy() - offset[c]
        for c in CALIBRATED
    }
    vals['mag'] = np.hypot(vals['dx'], vals['dy'])   # stored column -- must be rebuilt

    out = df.copy()
    for c, v in vals.items():
        out[[f'{c}_{i}' for i in range(99)]] = v

    return out

In [ ]:
BASE = {}
for p, sessions in participants.items():
    for s in sessions:
        sdir = session_dir(p, s)
        BASE[(p, s)] = add_flags(drop_unlabeled(sdir, load_session(p, s)))

OFFSETS = {k: rest_offset(df) for k, df in BASE.items()}

STALE_DONOR = {}
for p, sessions in participants.items():
    for i, s in enumerate(sessions):
        STALE_DONOR[(p, s)] = (p, sessions[i - 1])   # i=0 wraps to that participant's last session

for (p, s), (dp, ds) in STALE_DONOR.items():
    assert dp == p, (p, s, dp, ds)     # donor must never cross participants


def offset_for(mode, p, s):
    if mode == 'none':
        return None
    if mode == 'per_wear':
        return OFFSETS[(p, s)]
    if mode == 'stale':
        return OFFSETS[STALE_DONOR[(p, s)]]
    raise ValueError(f'unknown calibration mode: {mode!r}')


window_frames = {
    k: int(((df['trial_id'] == 0) & (df['phase'] == 'pre_cue') & (~df['braced'])).sum())
    for k, df in BASE.items()
}
assert min(window_frames.values()) >= 50, min(window_frames, key=window_frames.get)

for k, off in OFFSETS.items():
    for c in CALIBRATED:
        assert off[c].shape == (99,), (k, c, off[c].shape)
        assert not np.isnan(off[c]).any(), (k, c)

print(f"calibration window frames: min {min(window_frames.values())}, max {max(window_frames.values())}")

In [ ]:
def resting_level(df):
    """resting_level(df) -> float -- median resting displacement, scored on post-cue rest frames.

    Deliberately disjoint from the calibration window: scoring per_wear on its own
    reference data would read ~0 by construction and prove nothing.
    """
    m = (df['target_pct'] == 0) & (df['phase'] == 'post_cue')
    mag = df.loc[m, [f'mag_{i}' for i in range(99)]].to_numpy()
    return float(np.median(np.median(mag, axis=1)))


levels = {}
identity = {}

for mode in CALIBRATION_MODES:
    calibrated = {k: apply_calibration(df, offset_for(mode, *k)) for k, df in BASE.items()}
    levels[mode] = {k: resting_level(df) for k, df in calibrated.items()}
    identity[mode] = {k: df['trial_id'].to_numpy() for k, df in calibrated.items()}

for mode in CALIBRATION_MODES:
    for k in BASE:
        assert np.array_equal(identity[mode][k], identity['none'][k]), (mode, k)

spread = {mode: {} for mode in CALIBRATION_MODES}
for mode in CALIBRATION_MODES:
    for p, sessions in participants.items():
        vals = [levels[mode][(p, s)] for s in sessions]
        spread[mode][p] = max(vals) - min(vals)

# per_wear is expected to beat none for every participant, but is not asserted as a hard
# invariant: participant008 sessions 1 and 9 are a documented exception (localized-patch
# calibration limitation, see docs/participants/cross-participant-analysis.md #6), so one
# exception should not fail the whole notebook run.
calibration_pass = {p: spread['per_wear'][p] < spread['none'][p] for p in participants}
n_pass = sum(calibration_pass.values())
print(f"per_wear beats none: {n_pass}/{len(participants)} participants")
for p, passed in calibration_pass.items():
    if not passed:
        print(f"  exception: participant{p:03d} per_wear={spread['per_wear'][p]:.3f} none={spread['none'][p]:.3f}")

for mode in CALIBRATION_MODES:
    print(mode, {p: round(v, 2) for p, v in spread[mode].items()})

## Section 06 — Derived Channels

`strain_x/y`, `vx/vy`, `max_disp` via `core/features.py`. Runs on `CALIBRATION_MODE`-corrected data, so every derived channel inherits Section 05's correction.

In [ ]:
DERIVED_GRIDS = ('strain_x', 'strain_y', 'vx', 'vy')   # 99 columns each
DERIVED_SCALARS = ('max_disp',)                        # 1 column, whole grid


def add_channels(df, layout):
    """add_channels(df, layout) -> DataFrame -- derive strain/velocity/max_disp, flatten back to columns.

    layout[:, 0] / layout[:, 1] index the grid exactly as to_grid scattered it, so
    grid[:, rows, cols] undoes the scatter and returns marker_id order.
    """
    grids = {
        name: to_grid(df[[f'{name}_{i}' for i in range(99)]].to_numpy(), layout)
        for name in ('dx', 'dy', 'mag')
    }

    sx, sy = strain(grids['dx'], grids['dy'])
    vx, vy = velocity(
        grids['dx'], grids['dy'],
        df['timestamp_ms'].to_numpy(), df['trial_id'].to_numpy(),
    )

    derived = {'max_disp': max_disp(grids['mag'])}

    for name, grid in zip(DERIVED_GRIDS, (sx, sy, vx, vy)):
        flat = grid[:, layout[:, 0], layout[:, 1]]
        derived.update({f'{name}_{i}': flat[:, i] for i in range(99)})

    return pd.concat([df, pd.DataFrame(derived, index=df.index)], axis=1)   # one concat, not 397 inserts

In [ ]:
CAL_BASE = {k: apply_calibration(df, offset_for(CALIBRATION_MODE, *k)) for k, df in BASE.items()}
layout_arrays = {p: np.array(reference_layouts[p]) for p in participants}

n_base_cols = next(iter(BASE.values())).shape[1]
n_expected = n_base_cols + len(DERIVED_SCALARS) + len(DERIVED_GRIDS) * 99

velocity_nans = {}

for k, df in CAL_BASE.items():
    p, s = k
    layout = layout_arrays[p]
    full = add_channels(df, layout)

    assert full.shape == (len(df), n_expected), (k, full.shape, n_expected)

    # Round trip: scatter dx onto the grid, undo it, and the original must come back --
    # catches a transposed or mis-ordered layout, which would otherwise silently
    # scramble which marker each strain value belongs to.
    dx_raw = df[[f'dx_{i}' for i in range(99)]].to_numpy()
    dx_grid = to_grid(dx_raw, layout)
    assert np.array_equal(dx_grid[:, layout[:, 0], layout[:, 1]], dx_raw), k

    # Velocity is NaN at frame 0 and every trial boundary -- and nowhere else.
    trials = df['trial_id'].to_numpy()
    expected_nan = 1 + int((trials[1:] != trials[:-1]).sum())
    got_nan = int(full['vx_0'].isna().sum())
    assert got_nan == expected_nan, (k, got_nan, expected_nan)

    for name in ('strain_x', 'strain_y'):
        assert not full[[f'{name}_{i}' for i in range(99)]].isna().any().any(), (k, name)
    assert not full['max_disp'].isna().any(), k

    velocity_nans[k] = got_nan

print(f"columns: {n_base_cols} -> {n_expected}")
print(f"velocity NaN frames: {sum(velocity_nans.values())} of {sum(len(df) for df in BASE.values())}")

## Section 07 — Cache Write

69 × `features_table.csv` + `features_meta.json`, one per session folder. Written once so notebooks 03 onward never rerun Sections 0–6.

In [ ]:
SCHEMA_VERSION = 2         # v1 wrote 1 -- 'dirty' is a new field
V1_COMMIT = '0018b43'      # participant 001's v1 caches -- must not survive the rebuild

PER_MARKER_CHANNELS = list(CALIBRATED) + ['mag'] + list(DERIVED_GRIDS)


def git_stamp():
    """git_stamp() -> (short_hash, dirty) -- names the code that writes the cache.

    cwd must be ROOT_FOLDER. The notebook runs from notebooks/, and git answers for
    whichever folder you give it.

    A dirty tree makes the hash a lie about what ran, so the flag is recorded, not
    enforced. A block would stop every rebuild made mid-edit.
    """
    def git(*args):
        return subprocess.run(
            ['git', *args], cwd=ROOT_FOLDER,
            capture_output=True, text=True, check=True,
        ).stdout.strip()

    return git('rev-parse', '--short', 'HEAD'), bool(git('status', '--porcelain'))


def build_meta(p, s, df, stamp):
    """build_meta(p, s, df, stamp) -> dict -- the receipt written beside the CSV.

    n_rows and n_columns are read off df.shape. A constant would still report 806 on
    a file that wrote 400 rows and then died.
    """
    commit, dirty = stamp

    return {
        'schema_version': SCHEMA_VERSION,
        'built_at': datetime.now().isoformat(timespec='seconds'),
        'git_commit': commit,
        'dirty': dirty,
        'source': 'training_table.csv',
        'calibration_mode': CALIBRATION_MODE,
        'stale_donor': STALE_DONOR[(p, s)] if CALIBRATION_MODE == 'stale' else None,
        'per_marker_channels': PER_MARKER_CHANNELS,
        'scalar_channels': list(DERIVED_SCALARS),
        'n_rows': int(df.shape[0]),
        'n_columns': int(df.shape[1]),
    }


def write_session(p, s, df, stamp):
    """write_session(p, s, df, stamp) -> (csv_path, meta_path) -- both files, session folder only.

    index=False is not optional. Without it pandas writes the row index as a leading
    unnamed column, and the file comes back 807 wide instead of 806.
    """
    sdir = session_dir(p, s)
    csv_path = sdir / 'features_table.csv'
    meta_path = sdir / 'features_meta.json'

    df.to_csv(csv_path, index=False)
    with meta_path.open('w') as f:
        json.dump(build_meta(p, s, df, stamp), f, indent=2)

    return csv_path, meta_path

In [ ]:
""" Write all 69 caches in one run.

One run, one commit. Two batches put two hashes in the cache, and notebook 03's
git_commit assert then fails on data that is correct.

CAL_BASE is already calibrated. Do not call apply_calibration again here -- a second
pass subtracts the offset twice.

Expect about 3.4 GB across the 69 files. Print each write: a silent 3.4 GB run looks
the same as a hang.
"""
STAMP = git_stamp()

written = {}
skipped = []

for (p, s), df in CAL_BASE.items():
    sdir = session_dir(p, s)

    if not REBUILD_FEATURES and (sdir / 'features_table.csv').exists() \
            and (sdir / 'features_meta.json').exists():
        skipped.append((p, s))
        continue

    full = add_channels(df, layout_arrays[p])
    csv_path, _ = write_session(p, s, full, STAMP)
    written[(p, s)] = csv_path

    print(f"p{p:03d} s{s:<2} {full.shape[0]:>5} rows -> {csv_path.stat().st_size / 1e6:5.1f} MB")

total_mb = sum(c.stat().st_size for c in written.values()) / 1e6
print(f"\ncommit {STAMP[0]}{' (dirty tree)' if STAMP[1] else ''}, mode {CALIBRATION_MODE}")
print(f"{len(written)} written, {len(skipped)} skipped, {total_mb:.0f} MB on disk")

In [ ]:
# --- §7 acceptance ---
# Every check reads back off disk. A check against the frame just written cannot see
# a truncated write, a wrong folder, or the index column pandas adds on the way out.

def csv_shape(path):
    """csv_shape(path) -> (n_rows, n_cols) -- header and line count only, never the values.

    Reading every file in full is 3.4 GB at 7 participants and grows from there.
    """
    n_cols = pd.read_csv(path, nrows=0).shape[1]
    n_rows = sum(1 for _ in path.open('rb')) - 1     # minus the header line
    return n_rows, n_cols


metas = {}
for p, sessions in participants.items():
    for s in sessions:
        sdir = session_dir(p, s)
        assert (sdir / 'features_table.csv').exists(), (p, s)
        assert (sdir / 'features_meta.json').exists(), (p, s)

        with (sdir / 'features_meta.json').open() as f:
            metas[(p, s)] = json.load(f)

assert len(metas) == sum(len(v) for v in participants.values()), len(metas)

# One file read back in full. Only the file on disk can show the 807th column, and
# only the names catch a channel that was renamed or reordered.
probe = pd.read_csv(session_dir(1, 1) / 'features_table.csv')
assert probe.shape[1] == 806, probe.shape
assert list(probe.columns) == list(add_channels(CAL_BASE[(1, 1)], layout_arrays[1]).columns)

for k, meta in metas.items():
    assert (meta['n_rows'], meta['n_columns']) == csv_shape(session_dir(*k) / 'features_table.csv'), k
    assert meta['calibration_mode'] == CALIBRATION_MODE, k
    assert meta['git_commit'] != V1_COMMIT, k        # participant 001's v1 caches must be gone

# One shared commit is the assert. Equality with HEAD is only a print: a commit made
# after the write moves HEAD, and that is stale bookkeeping, not a corrupt cache.
cache_commits = {m['git_commit'] for m in metas.values()}
assert len(cache_commits) == 1, cache_commits
if len(participants) == FROZEN_PARTICIPANT_COUNT:
    assert sum(m['n_rows'] for m in metas.values()) == 264607

head_now = git_stamp()[0]
commit = next(iter(cache_commits))
print(f"§7 OK: {len(metas)} caches at commit {commit}, "
      f"{sum(m['n_rows'] for m in metas.values())} rows, {probe.shape[1]} columns")
if commit != head_now:
    print(f"note: HEAD is now {head_now} -- the cache predates it, rebuild to restamp")

## Section 08 — Index Table

Six identity columns for every discovered row (264,607 for the original 7-participant cohort). Never the feature columns.

In [ ]:
""" Build INDEX -- identity only, never a feature column.

Six columns for 264,607 rows is about 13 MB, against 3.41 GB for the full cache.
Section 9's folds and Section 11's assertions read identity and nothing else, so the
wide frame v1 built is not needed here and does not fit at 7 participants.

Read from BASE, not from the cache on disk. Fold membership must come from
pre-calibration identity, and BASE already holds it in memory.
"""

frames = []

for (p, s), df in BASE.items():
    smaller_df = df[['trial_id', 'target_pct', 'phase', 'to_cue']]
    frames.append(smaller_df.assign(participant_id=p, session_id=s))

INDEX_COLS = ['participant_id', 'session_id', 'trial_id', 'target_pct', 'phase', 'to_cue']

INDEX = pd.concat(frames).reset_index(drop=True)
INDEX = INDEX[INDEX_COLS]

print(f"{len(INDEX)} rows x {INDEX.shape[1]} cols, "
      f"{INDEX.memory_usage(deep=True).sum() / 1e6:.0f} MB")

In [ ]:
# --- §8 acceptance ---
INDEX_COLS = ['participant_id', 'session_id', 'trial_id', 'target_pct', 'phase', 'to_cue']

assert list(INDEX.columns) == INDEX_COLS, list(INDEX.columns)
assert len(INDEX) == sum(rows_after.values()), (len(INDEX), sum(rows_after.values()))

# A concat without reset_index leaves N copies of 0..n. Nothing raises, and every
# later .loc quietly pulls rows from multiple sessions at once.
assert INDEX.index.equals(pd.RangeIndex(len(INDEX))), INDEX.index[:5]

# Row counts survive the stack, session by session.
sizes = INDEX.groupby(['participant_id', 'session_id']).size()
assert len(sizes) == sum(len(v) for v in participants.values()), len(sizes)
assert sizes.to_dict() == rows_after, set(sizes.to_dict().items()) ^ set(rows_after.items())

n_trials = INDEX.groupby(['participant_id', 'session_id', 'trial_id']).ngroups

# Frozen-cohort regression guard -- exact totals only mean something while
# `participants` is still exactly the original FROZEN_PARTICIPANT_COUNT cohort.
if len(participants) == FROZEN_PARTICIPANT_COUNT:
    assert len(INDEX) == 264607, len(INDEX)
    assert n_trials == 2070, n_trials

# The identity spine must be complete. NaN belongs only where Sections 3 and 4 put it:
# to_cue and phase on the 14 uncued trials, and target_pct on participant 006
# session 4 trial 13, the orphan trial with no assigned level.
assert INDEX[['participant_id', 'session_id', 'trial_id']].notna().all().all()
assert int(INDEX['phase'].isna().sum()) == phase_unknown, INDEX['phase'].isna().sum()
assert int(INDEX['to_cue'].isna().sum()) == phase_unknown, INDEX['to_cue'].isna().sum()

print(f"§8 OK: {len(INDEX)} rows x {INDEX.shape[1]} cols, "
      f"{INDEX.memory_usage(deep=True).sum() / 1e6:.0f} MB, "
      f"{len(sizes)} sessions, {n_trials} trials")

## Section 09 — Folds

One session fold per session, grouped by participant (69 for the original 7-participant cohort); one subject fold per participant; and a within-mount trial split per participant.

In [ ]:
""" Build the fold manifest -- three families, from INDEX and participants.

Session folds and subject folds answer different questions and must not be mixed. A
held-out subject is also a held-out set of mounts, so a subject fold contains
everything a session fold contains, plus anatomy. The two families stay apart so a
reader can tell which evaluation a fold belongs to.

Nothing here reads a feature column. Fold membership comes from pre-calibration
identity, and that is all INDEX holds.

v1's flat ALL_SESSIONS is gone. Session 3 exists seven times now, once per person, so
a flat list of session numbers cannot say whose fold it is. Use participants[p].
"""

TEST_FRACTION = 0.2

# --- 9a  session folds: one per session, nested under its participant ---
SESSION_FOLDS = {}
for p in sorted(participants):
    folds = []
    for i, n in enumerate(participants[p]):
        folds.append({
            'fold': i,
            'test_session': n,
            'train_sessions': [m for m in participants[p] if m != n],
        })
    SESSION_FOLDS[p] = folds

print(f"9a: {sum(len(f) for f in SESSION_FOLDS.values())} session folds "
      f"across {len(SESSION_FOLDS)} participants")

# --- 9b  subject folds: 7, at the top level ---
SUBJECT_FOLDS = [
    {
        'fold': i,
        'test_participant': n,
        'train_participants': [m for m in sorted(participants) if m != n]
    }
    for i, n in enumerate(sorted(participants))
]

print(f"9b: {len(SUBJECT_FOLDS)} subject folds, "
      f"{len(SUBJECT_FOLDS[0]['train_participants'])} train participants each")

# --- 9c  trial split: 20% of each participant's trials held out ---
rng = np.random.default_rng(RANDOM_SEED)    # ONCE, before the loop. A fresh rng per
                                            # participant draws the same positions for
                                            # everybody and the split stops being random.
TRIAL_SPLIT = {}
for p in sorted(participants):
    rows = INDEX.loc[INDEX['participant_id'] == p, ['session_id', 'trial_id']]
    pairs = sorted((int(s), int(t)) for s, t in rows.drop_duplicates().to_numpy())

    order = rng.permutation(len(pairs))
    n_test = round(len(pairs) * TEST_FRACTION)

    # pairs is a list and order is an ndarray, so pairs[order[:n_test]] raises
    # TypeError -- take one position at a time. Sort both sides before storing, so
    # the manifest Section 10 writes reads the same on every run.
    test = sorted(pairs[i] for i in order[:n_test])
    train = sorted(pairs[i] for i in order[n_test:])

    TRIAL_SPLIT[p] = {
        'test_fraction': TEST_FRACTION,
        'stratified': False,   # protocol balance already fixes the levels: 5 x 6 reps
        'train': train,
        'test': test,
    }

print("9c: " + "  ".join(f"p{p:03d} {len(v['train'])}/{len(v['test'])}"
                         for p, v in TRIAL_SPLIT.items()))

In [ ]:
# --- §9 acceptance ---
# Not asserted, structural: fold membership comes from INDEX, which was built from
# BASE before any calibration. Calibration cannot move a row into another fold.

TARGET_LEVELS = [0, 5, 20, 50, 80]
TEST_FRACTION = 0.2

# Session folds -- one per session, and never across people.
expected_total_sessions = sum(len(v) for v in participants.values())
assert sum(len(f) for f in SESSION_FOLDS.values()) == expected_total_sessions

for p, folds in SESSION_FOLDS.items():
    assert len(folds) == len(participants[p]), (p, len(folds))
    assert [f['fold'] for f in folds] == list(range(len(folds))), p
    assert sorted(f['test_session'] for f in folds) == participants[p], p   # held out once each

    for f in folds:
        # The train set is this participant's OTHER sessions. A fold that trains on
        # another person still runs, still prints an R2, and the number means nothing.
        assert f['test_session'] not in f['train_sessions'], (p, f['fold'])
        assert sorted(f['train_sessions'] + [f['test_session']]) == participants[p], (p, f['fold'])

# Subject folds -- one per participant, with disjoint sides.
assert len(SUBJECT_FOLDS) == len(participants), len(SUBJECT_FOLDS)
assert sorted(f['test_participant'] for f in SUBJECT_FOLDS) == sorted(participants)

for f in SUBJECT_FOLDS:
    assert f['test_participant'] not in f['train_participants'], f['fold']
    assert sorted(f['train_participants'] + [f['test_participant']]) == sorted(participants), f['fold']

# Every held-out session must still carry all five levels, or a fold scores a model
# on a range it was never asked to predict. Verified off the raw CSVs 2026-08-25:
# all 69 sessions of the frozen cohort hold {0, 5, 20, 50, 80}; re-verified live here
# for whatever `participants` holds now.
for k, g in INDEX.groupby(['participant_id', 'session_id']):
    assert sorted(g['target_pct'].dropna().unique()) == TARGET_LEVELS, (k, g['target_pct'].unique())

# Trial split -- a true partition of that participant's trials, per participant.
# Assert the key set first. An empty TRIAL_SPLIT walks the loop below zero times,
# fires no assert, and prints "0 trials held out of 0" under a §9 OK banner.
assert sorted(TRIAL_SPLIT) == sorted(participants), sorted(TRIAL_SPLIT)

for p, split in TRIAL_SPLIT.items():
    rows = INDEX.loc[INDEX['participant_id'] == p, ['session_id', 'trial_id']]
    pairs = {(int(s), int(t)) for s, t in rows.drop_duplicates().to_numpy()}

    train = {tuple(x) for x in split['train']}
    test = {tuple(x) for x in split['test']}

    assert train | test == pairs, (p, len(train | test), len(pairs))   # nothing invented or lost
    assert not (train & test), (p, sorted(train & test)[:5])           # both sides is a leak
    assert round(len(test) / len(pairs), 2) == TEST_FRACTION, (p, len(test), len(pairs))

print(f"§9 OK: {sum(len(f) for f in SESSION_FOLDS.values())} session folds, "
      f"{len(SUBJECT_FOLDS)} subject folds, "
      f"{sum(len(v['test']) for v in TRIAL_SPLIT.values())} trials held out of "
      f"{sum(len(v['train']) + len(v['test']) for v in TRIAL_SPLIT.values())}")

## Section 10 — Split Manifest

Write `split_manifest.json` at schema_version 2. Keys participants by name string, never by int -- `json.dump` would silently turn an int key into a string that is not `"participant001"`.

Each session fold also carries `inner_val_sessions` -- 3 of that fold's training sessions, chosen to rotate so every session gets validation duty equally. Full reasoning: `docs/design/models/model-comparison-protocol.md` §3.

In [ ]:
""" Build split_manifest.json -- schema_version 2, keyed by participant.

Key type: "participant001" strings, matching v1 and matching how the participant
folders are already named on disk. Everything nested inside -- session numbers,
fold indices, trial ids -- stays plain int, exactly as SESSION_FOLDS, SUBJECT_FOLDS
and TRIAL_SPLIT already hold them.

excluded has no source in the notebook -- Section 1 discovers usable sessions by the
presence of training_table.csv and never records why one was missing. Written from a
literal, verified against the raw export before being trusted: participant 001
session 4's dynamometer.txt is 9 lines, all header, 0 data rows (session 5's, for
comparison, is 39,254 lines).
"""

EXCLUDED_REASONS = {
    (1, 4): "force export has no data rows",   # dynamometer.txt: 9 lines, 0 data rows
}


def participant_key(p):
    return f'participant{p:03d}'


# --- 10a  inner-3 rotation: which 3 of each fold's training sessions serve as
# inner validation, chosen so every session takes the turn equally.
#
# Shuffle each participant's own session list once under RANDOM_SEED, then walk
# it in a circle: the eligible trio for a fold is the next 3 sessions after the
# test session, wrapping past the end. In a circular walk of N sessions, each
# one is one of the "next 3" for exactly 3 different test sessions -- the
# balance is exact by construction, not by luck. Verified against the real
# session lists for all 7 participants (N=9 and N=10) before this cell was written.
#
# One rng, seeded once, participants visited in sorted order -- the same trap
# 9c already closed for TRIAL_SPLIT: a fresh rng per participant draws the same
# permutation positions for everybody and the rotation stops being independent.
rng = np.random.default_rng(RANDOM_SEED)

INNER_VAL_SESSIONS = {}
for p in sorted(participants):
    order = rng.permutation(participants[p]).tolist()
    n = len(order)
    trio_by_test_session = {
        order[i]: [order[(i + 1) % n], order[(i + 2) % n], order[(i + 3) % n]]
        for i in range(n)
    }
    INNER_VAL_SESSIONS[p] = {
        fold['fold']: trio_by_test_session[fold['test_session']]
        for fold in SESSION_FOLDS[p]
    }


MANIFEST_PATH = CURRENT_FOLDER / 'split_manifest.json'
STAMP = git_stamp()

manifest_participants = {}
for p in sorted(participants):
    excluded = {
        str(s): reason for (pp, s), reason in EXCLUDED_REASONS.items() if pp == p
    }
    rows = INDEX.loc[INDEX['participant_id'] == p]

    manifest_participants[participant_key(p)] = {
        'sessions': participants[p],
        'excluded': excluded,
        'n_rows': int(len(rows)),
        'n_trials': int(rows.groupby(['session_id', 'trial_id']).ngroups),
        'session_folds': [
            {**fold, 'inner_val_sessions': INNER_VAL_SESSIONS[p][fold['fold']]}
            for fold in SESSION_FOLDS[p]
        ],
        'trial_split': {
            'test_fraction': TRIAL_SPLIT[p]['test_fraction'],
            'stratified': TRIAL_SPLIT[p]['stratified'],
            'train': [list(x) for x in TRIAL_SPLIT[p]['train']],
            'test': [list(x) for x in TRIAL_SPLIT[p]['test']],
        },
    }

manifest_subject_folds = [
    {
        'fold': f['fold'],
        'test_participant': participant_key(f['test_participant']),
        'train_participants': [participant_key(m) for m in f['train_participants']],
    }
    for f in SUBJECT_FOLDS
]

MANIFEST = {
    'schema_version': SCHEMA_VERSION,
    'built_at': datetime.now().isoformat(timespec='seconds'),
    'git_commit': STAMP[0],
    'dirty': STAMP[1],
    'random_seed': RANDOM_SEED,
    'calibration_mode': CALIBRATION_MODE,
    'participants': manifest_participants,
    'subject_folds': manifest_subject_folds,
    'totals': {
        'participants': len(manifest_participants),
        'sessions': sum(len(v['sessions']) for v in manifest_participants.values()),
        'session_folds': sum(len(v['session_folds']) for v in manifest_participants.values()),
        'subject_folds': len(manifest_subject_folds),
    },
}

with MANIFEST_PATH.open('w') as f:
    json.dump(MANIFEST, f, indent=2)

print(f"{MANIFEST_PATH.name}: {MANIFEST_PATH.stat().st_size / 1e3:.0f} kB, "
      f"commit {STAMP[0]}{' (dirty tree)' if STAMP[1] else ''}, "
      f"{MANIFEST['totals']['participants']} participants, "
      f"{MANIFEST['totals']['sessions']} sessions, "
      f"{MANIFEST['totals']['session_folds']} session folds, "
      f"{MANIFEST['totals']['subject_folds']} subject folds")


## Section 11 — Assertions

Read `split_manifest.json` back off disk. Assert it matches `SESSION_FOLDS`, `SUBJECT_FOLDS`, and `TRIAL_SPLIT` exactly, and that every value survived the round trip through JSON. Also assert the inner-3 rotation: exactly 3 eligible sessions per fold, all inside that fold's training sessions, and every session takes the validation turn exactly 3 times across its participant's folds.

In [ ]:
# --- §11 acceptance ---
# Every check reads the file back off disk. The dict just written cannot show a
# tuple that turned into a list, an int key that turned into "1", or a write that
# never landed on the path you meant.

PARTICIPANT_KEYS = {'sessions', 'excluded', 'n_rows', 'n_trials', 'session_folds', 'trial_split'}
PIDS = [participant_key(p) for p in sorted(participants)]

with MANIFEST_PATH.open() as f:
    MF = json.load(f)

assert set(MF) == {'schema_version', 'built_at', 'git_commit', 'dirty', 'random_seed',
                    'calibration_mode', 'participants', 'subject_folds', 'totals'}, set(MF)

assert MF['schema_version'] == SCHEMA_VERSION, MF['schema_version']
assert MF['random_seed'] == RANDOM_SEED, MF['random_seed']
assert MF['calibration_mode'] == CALIBRATION_MODE, MF['calibration_mode']
assert isinstance(MF['dirty'], bool), MF['dirty']
datetime.fromisoformat(MF['built_at'])                 # raises if not ISO

# A write to the wrong path leaves the v1 manifest sitting there, and it parses
# cleanly. This line and the schema_version above are what catch that.
assert MF['git_commit'] != V1_COMMIT, MF['git_commit']

# Key set before the loop. An empty participants dict walks the loop zero times,
# fires no assert, and prints an OK banner -- the hole §9 already found once.
assert list(MF['participants']) == PIDS, list(MF['participants'])

n_rows_seen = 0
n_trials_seen = 0

for p in sorted(participants):
    rec = MF['participants'][participant_key(p)]
    assert set(rec) == PARTICIPANT_KEYS, (p, set(rec))
    assert rec['sessions'] == participants[p], (p, rec['sessions'])

    # Independent of the build: every session folder on disk is either usable or
    # excluded, never neither. json turns an int key into a string on the way out,
    # so the excluded keys read back as "4", never 4.
    pdir = next(recordings_dir.glob(f'participant{p:03d}_*'))
    on_disk = sorted(session_num(folder) for folder in pdir.glob('session*'))
    excluded = [int(s) for s in rec['excluded']]
    assert sorted(rec['sessions'] + excluded) == on_disk, (p, rec['sessions'], excluded, on_disk)

    for s, reason in rec['excluded'].items():
        assert isinstance(reason, str) and reason, (p, s, reason)

    rows = INDEX.loc[INDEX['participant_id'] == p]
    assert rec['n_rows'] == len(rows), (p, rec['n_rows'], len(rows))
    assert rec['n_trials'] == rows.groupby(['session_id', 'trial_id']).ngroups, p
    assert rec['n_trials'] == 30 * len(rec['sessions']), (p, rec['n_trials'])   # 5 levels x 6 reps

    # np.int64 cannot reach the file -- json.dump raises on it, so the write already
    # tested that. np.float64 CAN, since it subclasses float, so a stray float()
    # would write 28261.0 and nothing would complain. This line is for that case.
    assert type(rec['n_rows']) is int and type(rec['n_trials']) is int, (p, type(rec['n_rows']))

    n_rows_seen += rec['n_rows']
    n_trials_seen += rec['n_trials']

    # §9 already proved these folds correct in memory. This is the round trip only.
    # Compared field-by-field, not by dict equality, because the manifest's
    # session_folds now carries one key SESSION_FOLDS never had: inner_val_sessions.
    assert len(rec['session_folds']) == len(SESSION_FOLDS[p]), p
    inner_val_counts = {}
    for fold, base in zip(rec['session_folds'], SESSION_FOLDS[p]):
        assert fold['fold'] == base['fold'], p
        assert fold['test_session'] == base['test_session'], p
        assert fold['train_sessions'] == base['train_sessions'], p
        assert fold['test_session'] not in fold['train_sessions'], (p, fold['fold'])

        # Inner-3 rotation: exactly 3 unique eligible sessions, all drawn from
        # this fold's own training pool -- never the outer test session.
        inner = fold['inner_val_sessions']
        assert len(inner) == 3 and len(set(inner)) == 3, (p, fold['fold'], inner)
        assert set(inner) <= set(fold['train_sessions']), (p, fold['fold'], inner)
        for s in inner:
            inner_val_counts[s] = inner_val_counts.get(s, 0) + 1

    # The balance guarantee: in a circular walk of N sessions, each one is the
    # "next 3" of exactly 3 different test sessions -- so across a participant's
    # full set of folds, every session must serve inner validation exactly 3
    # times, not "roughly" 3.
    assert set(inner_val_counts) == set(rec['sessions']), (p, inner_val_counts)
    assert set(inner_val_counts.values()) == {3}, (p, inner_val_counts)

    split = rec['trial_split']
    assert set(split) == {'test_fraction', 'stratified', 'train', 'test'}, (p, set(split))
    assert split['stratified'] is False, p
    assert split['test_fraction'] == TEST_FRACTION, (p, split['test_fraction'])

    # json has no tuple -- TRIAL_SPLIT holds (session_id, trial_id) tuples, the file
    # holds [session_id, trial_id] lists. Compare list to list or every trial differs.
    assert split['train'] == [list(x) for x in TRIAL_SPLIT[p]['train']], p
    assert split['test'] == [list(x) for x in TRIAL_SPLIT[p]['test']], p
    assert all(len(x) == 2 for x in split['train'] + split['test']), p   # never a bare trial id

    train = {tuple(x) for x in split['train']}
    test = {tuple(x) for x in split['test']}
    assert not (train & test), (p, sorted(train & test)[:5])
    assert len(train) + len(test) == rec['n_trials'], (p, len(train), len(test))

assert n_rows_seen == len(INDEX), n_rows_seen
if len(participants) == FROZEN_PARTICIPANT_COUNT:
    assert n_rows_seen == 264607, n_rows_seen
    assert n_trials_seen == 2070, n_trials_seen

# Subject folds -- one per participant, at the top level, disjoint sides.
assert len(MF['subject_folds']) == len(participants), len(MF['subject_folds'])
assert [f['fold'] for f in MF['subject_folds']] == list(range(len(participants)))
assert sorted(f['test_participant'] for f in MF['subject_folds']) == sorted(PIDS)

for fold in MF['subject_folds']:
    assert set(fold) == {'fold', 'test_participant', 'train_participants'}, fold['fold']

    # Half a conversion is the real risk: participant keys turned into strings and
    # these values left as ints. Both sides must already be the participant000 form.
    assert fold['test_participant'] in PIDS, fold
    assert set(fold['train_participants']) <= set(PIDS), fold
    assert fold['test_participant'] not in fold['train_participants'], fold['fold']
    assert sorted(fold['train_participants'] + [fold['test_participant']]) == sorted(PIDS), fold['fold']

# Totals cross-checked against the file's own contents, never a literal alone.
assert set(MF['totals']) == {'participants', 'sessions', 'session_folds', 'subject_folds'}, set(MF['totals'])
assert MF['totals']['participants'] == len(MF['participants']) == len(participants)
assert MF['totals']['sessions'] == sum(len(r['sessions']) for r in MF['participants'].values()) == sum(len(v) for v in participants.values())
assert MF['totals']['session_folds'] == sum(len(r['session_folds']) for r in MF['participants'].values()) == sum(len(v) for v in participants.values())
assert MF['totals']['subject_folds'] == len(MF['subject_folds']) == len(participants)

print(f"§11 OK: {MANIFEST_PATH.stat().st_size / 1e3:.0f} kB, "
      f"schema {MF['schema_version']}, commit {MF['git_commit']}"
      f"{' (dirty tree)' if MF['dirty'] else ''}, "
      f"{MF['totals']['sessions']} sessions, "
      f"{MF['totals']['session_folds']} session folds, "
      f"{MF['totals']['subject_folds']} subject folds, "
      f"{sum(len(r['trial_split']['test']) for r in MF['participants'].values())} trials held out")

## Section 12 — Summary

**Bottom line:** 8 participants, 79 usable sessions, 302,047 rows, 806 columns, split three ways for the SO4 evaluation ladder.

### Discovery

| Participant | Sessions used | Raw rows | Note |
|---|---|---|---|
| 001 | 9 of 10 | 28,261 | session 4 has no force export; trial (2, 13) never cued |
| 002 | 10 | 43,941 | negative resting force, behavioural |
| 003 | 10 | 38,233 | sessions 6, 7, 10 coupled poorly, kept |
| 004 | 10 | 38,669 | session 10 camera stopped at 141 s, Biopac ran to 149 s |
| 005 | 10 | 40,622 | — |
| 006 | 10 | 37,874 | — |
| 007 | 10 | 37,462 | MVC file is participant 006's, kept by decision |
| 008 | 10 | 37,854 | — |
| **Total** | **79** | **302,916** | 1 distinct marker-grid layout across all 8 |

### Pipeline funnel

| Stage | Rows | Columns |
|---|---|---|
| Raw, loaded (§2) | 302,916 | 405 |
| After label filter (§3) | 302,047 | 405 |
| After derived channels (§6) | 302,047 | 806 |

869 rows dropped dataset-wide for a missing force label (up from 455 at n = 7; participant008 contributes the remaining 414). 16 uncued trials kept and flagged (`to_cue` and `phase` both NaN).

### Calibration contrast

Median resting displacement, max − min across a participant's own sessions. Lower is better; `per_wear` is the mode carried into the cache.

| Participant | none | stale | per_wear |
|---|---|---|---|
| 001 | 4.17 | 3.41 | 1.80 |
| 002 | 1.85 | 1.60 | 0.79 |
| 003 | 4.15 | 4.66 | 1.60 |
| 004 | 2.15 | 3.64 | 1.31 |
| 005 | 2.38 | 2.90 | 1.42 |
| 006 | 1.67 | 1.84 | 0.94 |
| 007 | 6.13 | 4.96 | 1.19 |
| 008 | 0.97 | 1.18 | 1.05 |

`per_wear` beats `none` in 7 of 8 participants (reported per-participant, not hard-asserted, in §5 — participant008 is a documented exception, driven by two sessions with a localized-patch calibration limitation; see `docs/design/calibration/so4-calibration-protocol.md` §7b). `CALIBRATION_MODE = "per_wear"` stands as the decision.

### Cache and manifest

| Artifact | Size | Commit | Schema |
|---|---|---|---|
| `features_table.csv` × 79 | 3891 MB | `e0ce786` | 2 |
| `split_manifest.json` | 174 kB | `e0ce786` | 2 |

Both were written with a dirty working tree at `e0ce786` — the commit names the code that ran, not a clean checkout.

### Folds — the SO4 evaluation ladder

| Fold family | Count | Held out |
|---|---|---|
| Session folds | 79 | one session per fold, within-participant |
| Subject folds | 8 | one participant per fold |
| Trial split | 474 of 2,370 | 20% per participant, not stratified |

| Participant | Train trials | Test trials |
|---|---|---|
| 001 | 216 | 54 |
| 002–008 (each) | 240 | 60 |

Not stratified: protocol balance (5 target levels × 6 reps) already fixes the class balance.